In [1]:
import os
import pickle
from pathlib import Path
import numpy as np

# import required module
import sys
 
# append the path of the
# parent directory
sys.path.append("..")
 
# import method from sibling 
# module
from data_utils import save_samples_as_cifs, save_reconstructions_as_cifs, visualize_trajectory
from utils import retrieve_artifacts_by_name


import env

# Load environment variables
env.load_envs()

# Set the cwd to the project root
PROJECT_ROOT: Path = Path(env.get_env("PROJECT_ROOT"))
assert (
    PROJECT_ROOT.exists()
), "You must configure the PROJECT_ROOT environment variable in a .env file!"

os.chdir(PROJECT_ROOT)

cwd = os.getcwd()

c:\Users\dglav\Anaconda3\envs\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Parse samples

In [ ]:
experiment_name = "samples-sample_only_MOR_mean"

# artifact_files = retrieve_artifacts_by_name(experiment_name, artifact_type='dataset', project='zeogen', entity='glafk')

# for file in artifact_files:
#     if "samples" in file:
#         with open(file, "rb") as f:
#             samples = pickle.load(f)


samples_file = "samples-sample_only_MOR_mean.pickle"

with open(f"./samples/{samples_file}", "rb") as f:
    samples = pickle.load(f)
    samples = samples[0]

print(len(samples))
for sample in samples:
    print(sample.keys())
    print(sample["num_atoms"])
    print(len(sample["frac_coords"]))
    print(sample["frac_coords"].shape)
    print(len(sample["all_frac_coords"]))


save_samples_as_cifs(samples, os.path.join(cwd, f"parsed_samples/samples_{experiment_name}"), save_trajectory=True, downsample_trajectory=True, downsample_frame_rate=10)


### Visualize trajectories

In [ ]:
with open("./samples/samples_low_noise.pickle", "rb") as f:
    samples = pickle.load(f)

print(samples.keys())
print(samples["all_frac_coords"].shape)
samples["atom_types"] = samples["atom_types"].cpu()
samples["angles"] = samples["angles"].cpu()
samples["lengths"] = samples["lengths"].cpu()
samples["num_atoms"] = samples["num_atoms"].cpu()
samples["frac_coords"] = samples["frac_coords"].cpu()
samples["all_frac_coords"] = samples["all_frac_coords"].cpu()

# Split atom types
split_atom_types = np.split(samples["atom_types"], np.cumsum(samples["num_atoms"])[:-1])

# Split fractional coordinates
split_frac_coords = np.split(samples["frac_coords"], np.cumsum(samples["num_atoms"])[:-1])


trajectories = [samples["all_frac_coords"][:, i*48:(i+1)*48] for i in range(50)]

individual_samples = []
for i in range(len(samples["num_atoms"])):
    individual_samples.append({"atom_types": split_atom_types[i], "frac_coords": split_frac_coords[i], "lengths": samples["lengths"][i], "angles": samples["angles"][i]})

print(trajectories[0].shape)
print(trajectories[0].min(), trajectories[0].max())
visualize_trajectory(trajectories[0][-100:,0,:], individual_samples[0]["lengths"])

## Parse reconstructions

In [3]:
experiment_name = "reconstructions-overfit_MOR_10_epochs"

# artifact_files = retrieve_artifacts_by_name(experiment_name, artifact_type='dataset', project='zeogen', entity='glafk')

# for file in artifact_files:
#     if "reconstructions" in file:
#         with open(file, "rb") as f:
#             reconstructions = pickle.load(f)

reconstructions_file = "reconstructions-reconstruct_overfit_MOR_10_epochs.pickle"

with open(f"./reconstructions/{reconstructions_file}", "rb") as f:
    reconstructions = pickle.load(f)

recon_path = os.path.join(cwd, f"parsed_reconstructions/reconstructions_{experiment_name}")
save_reconstructions_as_cifs(reconstructions, recon_path, save_trajectory=True, downsample_trajectory=True, downsample_frame_rate=5)

{'zd': array([[ 0.14208536, -0.25718588,  0.79684335,  0.00856692, -0.2045438 ,
         0.26749647,  0.30188832, -0.82089746,  1.142238  , -0.88672924,
         0.07609648,  0.11567691, -1.1879597 , -0.05488989,  0.19698656,
        -0.00503716, -1.412152  , -0.11393082, -0.694981  ,  0.04434615,
         0.01582992,  0.45608762,  0.46480948, -0.8294504 ,  0.7725615 ,
         0.7004671 ,  0.575534  , -0.7367938 , -0.36453304, -0.14100525,
        -0.19973423, -0.85151535, -0.5885455 ,  0.853682  ,  0.71167713,
        -0.71821636,  0.3134591 ,  0.20580663,  0.3303703 , -1.4235065 ,
         0.05824096,  0.21148627,  0.32736528, -0.06761414, -0.5652722 ,
        -0.47617993,  0.34840888,  0.5493834 ,  0.15633774,  0.5756832 ,
        -0.18145818, -0.42581606, -0.9298809 ,  0.12212852,  0.21534795,
        -0.42463222, -0.80599153, -0.44614527, -0.37999395,  0.2748516 ,
         0.29980737,  0.3591922 ,  0.09820242,  0.3968924 , -0.8979486 ,
        -0.5210258 , -0.4469869 ,  0.1674971

## Parse reconstruction ground truth

In [4]:
experiment_name = "reconstructions-overfit_MOR_10_epochs"

# artifact_files = retrieve_artifacts_by_name(experiment_name, artifact_type='dataset', project='zeogen', entity='glafk')

# for file in artifact_files:
#     if "reconstructions" in file and "gt" in file:
#         with open(file, "rb") as f:
#             reconstructions = pickle.load(f)

reconstructions_file_gt = "reconstructions-reconstruct_overfit_MOR_10_epochs_gt.pickle"

with open(f"./reconstructions/{reconstructions_file_gt}", "rb") as f:
    reconstructions_gt = pickle.load(f)

recon_path = os.path.join(cwd, f"parsed_reconstructions/reconstructions_{experiment_name}_gt")
save_reconstructions_as_cifs(reconstructions_gt, recon_path, ground_truth=True)

DataBatch(edge_index=[2, 192], frac_coords=[48, 3], atom_types=[48], lengths=[1, 3], angles=[1, 3], to_jimages=[192, 3], num_atoms=[1], num_bonds=[1], num_nodes=48, zeolite_code=[1], zeolite_code_enc=[1], hoa=[1, 1], hoa_mu=[1, 1], hoa_std=[1, 1], norm_hoa=[1, 1], batch=[48], ptr=[2])
['num_atoms', 'frac_coords', 'batch', 'zeolite_code_enc', 'norm_hoa', 'angles', 'to_jimages', 'hoa_mu', 'lengths', 'num_nodes', 'atom_types', 'hoa_std', 'num_bonds', 'ptr', 'hoa', 'edge_index', 'zeolite_code']
torch.Size([1, 3])
Saving to C:\TUE\Thesis\zeogen\parsed_reconstructions/reconstructions_reconstructions-overfit_MOR_10_epochs_gt\reconstruction_1_gt.cif.
DataBatch(edge_index=[2, 192], frac_coords=[48, 3], atom_types=[48], lengths=[1, 3], angles=[1, 3], to_jimages=[192, 3], num_atoms=[1], num_bonds=[1], num_nodes=48, zeolite_code=[1], zeolite_code_enc=[1], hoa=[1, 1], hoa_mu=[1, 1], hoa_std=[1, 1], norm_hoa=[1, 1], batch=[48], ptr=[2])
['num_atoms', 'frac_coords', 'batch', 'zeolite_code_enc', 'norm